In [1]:
!pip install syncode

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.1/206.1 kB 21.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 133.0 MB/s eta 0:00:00
  Created wheel for syncode: filename=syncode-0.4.16-py3-none-any.whl size=237009 sha256=459d9a4d5532a3c56704d0fb946808dcc5b87ded456f042ac21eb342d7984741
  Stored in directory: /root/.cache/pip/wheels/3f/a5/9b/9f6e1723cd27ffd3feea8185b6d3cd8578d22e8e240dc3ca9d
Successfully built syncode
  Attempting uni

In [19]:
from syncode import Syncode
import re

In [44]:
grammar = r"""
start: NUMBER OP NUMBER
OP: "+" | "-" | "*" | "/"
NUMBER: /-?[0-9]{1,10}(\.[0-9]{0,10})?/

%import common.WS
%ignore WS
"""
syn_llm = Syncode(model="microsoft/phi-2", grammar=grammar,
                  parse_output_only=True, max_new_tokens=20)

[2026-10-27 13:20:05,682-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16
[2026-10-27 13:20:05,986-accelerate.utils.modeling] - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[2026-10-27 13:20:28,297-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_strict_4284484130_50257.pkl exist: False
[2026-10-27 13:20:28,298-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and custom, may take more than 10 minutes. Caching at /content/cache/mask_stores/CodeGenTokenizerFast/grammar_strict_4284484130_50257.pkl.
[2026-10-27 13:20:28,838-syncode.mask_store.fsm_set] - 3 FSMs with 31 states initialized in 0.00 seconds
[2026-10-27 13:20:28,839-syncode.mask_store.mask_store] - Ignore whitespace is True
[2026-10-27 13:20:28,842-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 9 to 5


100%|██████████| 31/31 [00:09<00:00,  3.28it/s]

[2026-10-27 13:20:38,315-syncode.mask_store.mask_store] - Time taken to create mask store: 10.00 seconds


In [45]:
examples = """Extract the two operands and the operator.
Query: What is 99. multiplied by -7.676?
Answer: 99.0 * -7.676
Query: What is 5 plus -7.16?
Answer: 5.0 + -7.16
Query: What is 9911.01 divided by 0.1?
Answer: 9911.01 / 0.1
Query: What is 9.01 subtracted by 4.0?
Answer: 9.01 - 4.0
"""

In [54]:
def run_syncode(textual_query):
    prompt = examples + f"Query: {textual_query}\nAnswer:"
    out = syn_llm.infer(prompt)[0].strip()
    m = re.fullmatch(r"(-?[0-9]+(?:\.[0-9]*)?)\s*([+\-*/])\s*(-?[0-9]+(?:\.[0-9]*)?)", out)
    a = float(m.group(1))
    op = m.group(2)
    b = float(m.group(3))
    if op == "/" and b == 0:
      result = float("nan") if a == 0 else float("inf") * (1 if a > 0 else -1)
    else:
      result = {"+": a + b, "-": a - b, "*": a * b, "/": a / b}[op]
    return ([a, b], op, result)

In [55]:
for q in ["What is 327. multiplied by 11.0?",
          "What is 45.1 plus 23.54?",
          "What is 120.4 divided by 4.0?"]:
    print(q, "->", run_syncode(q))

What is 327. multiplied by 11.0? -> ([327.0, 11.0], '*', 3597.0)
What is 45.1 plus 23.54? -> ([45.1, 23.54], '+', 68.64)
What is 120.4 divided by 4.0? -> ([120.4, 4.0], '/', 30.1)


In [49]:
for q in ["What is 2 multiplied by 3?",
          "What is 99 times 7.676?",
          "What is 5.51 plus 7.1?",
          "What is -0.1 add -23?",
          "What is 9911.01 divided by -0.1?",
          "What is 9.01 subtracted by 4.0?",
          "What is 9.5 minus -2.25?"]:
    print(q, "->", run_syncode(q))

What is 2 multiplied by 3? -> ([2.0, 3.0], '*', 6.0)
What is 99 times 7.676? -> ([99.0, 7.676], '*', 759.924)
What is 5.51 plus 7.1? -> ([5.51, 7.1], '+', 12.61)
What is -0.1 add -23? -> ([-0.1, -23.0], '+', -23.1)
What is 9911.01 divided by -0.1? -> ([9911.01, -0.1], '/', -99110.09999999999)
What is 9.01 subtracted by 4.0? -> ([9.01, 4.0], '-', 5.01)
What is 9.5 minus -2.25? -> ([9.5, -2.25], '-', 11.75)


In [51]:
for q in ["What is 3.5 * 2?",
          "What is 3.5 x 2?",
          "What is 12 / 4?",
          "What is the sum of 3.2 and 4.8",
          "What is the product of 6.0 and 1.5?",
          "What is the difference between 10.0 and 4.5?",
          "Calculate 4.2 plus 1.1."]:
    print(q, "->", run_syncode(q))

What is 3.5 * 2? -> ([3.5, 2.0], '*', 7.0)
What is 3.5 x 2? -> ([3.5, 2.0], '*', 7.0)
What is 12 / 4? -> ([12.0, 4.0], '/', 3.0)
What is the sum of 3.2 and 4.8 -> ([3.2, 4.8], '+', 8.0)
What is the product of 6.0 and 1.5? -> ([6.0, 1.5], '*', 9.0)
What is the difference between 10.0 and 4.5? -> ([10.0, 4.5], '-', 5.5)
Calculate 4.2 plus 1.1. -> ([4.2, 1.1], '+', 5.300000000000001)
